## LeNet5

Outputs

- `models/lenet5_quantized.tflite` INT8 input/output, per-tensor; MicroFlow-compatible

Keep `epochs` small if you only need artifacts quickly.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import tensorflow as tf

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

try:
    tf.get_logger().setLevel("ERROR")
except Exception:
    pass

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OUT_TFLITE_QUANT = MODELS_DIR / "lenet5_quantized.tflite"

print("OUT_TFLITE_QUANT:", OUT_TFLITE_QUANT)

In [ ]:
def build_lenet5() -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(28, 28, 1), batch_size=1, name="input")
    x = tf.keras.layers.Conv2D(6, (5, 5), activation="relu", padding="valid")(inputs)
    x = tf.keras.layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))(x)
    x = tf.keras.layers.Conv2D(16, (5, 5), activation="relu", padding="valid")(x)
    x = tf.keras.layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))(x)
    x = tf.keras.layers.Reshape((256,), name="flatten_256")(x)
    x = tf.keras.layers.Dense(120, activation="relu", name="fc1")(x)
    x = tf.keras.layers.Dense(84, activation="relu", name="fc2")(x)
    logits = tf.keras.layers.Dense(10, name="fc3")(x)
    outputs = tf.keras.layers.Softmax(name="softmax")(logits)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="lenet5")

model = build_lenet5()
model.summary()

In [ ]:

epochs = 2
batch_size = 128

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = (x_train.astype(np.float32) / 255.0)[..., None]
x_test = (x_test.astype(np.float32) / 255.0)[..., None]

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose=2,
)

print("Test accuracy:", model.evaluate(x_test, y_test, verbose=0)[1])

In [ ]:
def representative_data_gen():
    for i in range(200):
        yield [x_train[i : i + 1].astype(np.float32)]

int8_converter = tf.lite.TFLiteConverter.from_keras_model(model)
int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
int8_converter.representative_dataset = representative_data_gen
int8_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
int8_converter.target_spec.supported_types = [tf.int8]
int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

for attr in (
    "_experimental_disable_per_channel",
    "_experimental_disable_per_channel_quantization",
):
    if hasattr(int8_converter, attr):
        setattr(int8_converter, attr, True)

for attr in ("experimental_new_quantizer", "_experimental_new_quantizer"):
    if hasattr(int8_converter, attr):
        setattr(int8_converter, attr, False)

int8_tflite = int8_converter.convert()
OUT_TFLITE_QUANT.write_bytes(int8_tflite)

int8_interp = tf.lite.Interpreter(model_path=str(OUT_TFLITE_QUANT), experimental_delegates=[])
int8_interp.allocate_tensors()
in_info = int8_interp.get_input_details()[0]
out_info = int8_interp.get_output_details()[0]

print("wrote", OUT_TFLITE_QUANT, "bytes=", OUT_TFLITE_QUANT.stat().st_size)
print("int8 input:", in_info["dtype"], in_info["shape"], "quant=", in_info["quantization"])
print("int8 output:", out_info["dtype"], out_info["shape"], "quant=", out_info["quantization"])

per_channel = []
for td in int8_interp.get_tensor_details():
    qp = td.get("quantization_parameters") or {}
    scales = qp.get("scales")
    if isinstance(scales, np.ndarray) and scales.size > 1:
        per_channel.append((td.get("name", "?"), int(scales.size)))

if per_channel:
    print("Warning: per-channel quantization detected (first 20):")
    for name, n in per_channel[:20]:
        print(" -", name, "scales=", n)
else:
    print("Per-channel quantization check: OK (all tensors per-tensor or unquantized)")
